In [1]:
!pip install torch-scatter -f https://data.pyg.org/whl/torch-2.11.0+cu128.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-2.11.0+cu128.html
!pip install torch-geometric

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html


In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import warnings
import altair as alt
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import NeighborLoader

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.__version__)
print(torch.version.cuda)


%env JOBLIB_TEMP_FOLDER=/tmp


True
Tesla T4
2.11.0+cu128
12.8
env: JOBLIB_TEMP_FOLDER=/tmp


Mounted at /content/drive


In [4]:
# Detect environment
try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    IS_COLAB = False

# Root paths
if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")

Running on Google Colab
Dataset path: /content/drive/MyDrive/minor-thesis/dataset


In [5]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
X = train.drop(columns=["isFraud"])
y = train["isFraud"].values

x = torch.tensor(X.values, dtype=torch.float)
y = torch.tensor(y, dtype=torch.long)

In [7]:
uid_counts = train["uid"].value_counts()

print(uid_counts.describe())
print("Largest uid group:", uid_counts.max())
print(uid_counts.nlargest(20))

count    14845.000000000000000
mean        39.780397440215559
std        298.099311552955612
min          1.000000000000000
25%          1.000000000000000
50%          4.000000000000000
75%         12.000000000000000
max      14112.000000000000000
Name: count, dtype: float64
Largest uid group: 14112
uid
14418    14112
5542     10332
6769     10312
13057     8844
4766      7918
2576      7079
11414     6766
2418      6760
8719      6126
13058     6047
12773     5325
610       5155
2706      5110
9005      4604
7646      4197
5178      3973
5781      3914
8788      3864
5722      3739
1151      3677
Name: count, dtype: int64


In [8]:
K = 5
edge_list = []

for _, group in train.groupby("uid"):
    group = group.sort_values("TransactionDT")
    idx = group.index.to_list()

    n = len(idx)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, min(i + K + 1, n)):
            edge_list.append([idx[i], idx[j]])
            edge_list.append([idx[j], idx[i]])

edge_index_uid = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index_uid.shape)

torch.Size([2, 5573580])


In [9]:
K = 5
edge_list = []

for _, group in train.groupby("uid2"):
    group = group.sort_values("TransactionDT")
    idx = group.index.to_list()

    n = len(idx)
    if n < 2:
        continue

    for i in range(n):
        for j in range(i + 1, min(i + K + 1, n)):
            edge_list.append([idx[i], idx[j]])
            edge_list.append([idx[j], idx[i]])

edge_index_uid2 = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

print(edge_index_uid2.shape)

torch.Size([2, 4227114])


In [10]:
def graph_statistics(edge_index, num_nodes, name):
    degree = torch.bincount(edge_index[0], minlength=num_nodes)

    print(f"===== {name} =====")
    print(f"Nodes           : {num_nodes:,}")
    print(f"Directed edges  : {edge_index.shape[1]:,}")
    print(f"Average degree  : {degree.float().mean():.2f}")
    print(f"Maximum degree  : {degree.max().item():,}")
    print(f"Isolated nodes  : {(degree == 0).sum().item():,}")
    print()

In [11]:
graph_statistics(edge_index_uid, len(train), "G1 (uid)")
graph_statistics(edge_index_uid2, len(train), "G2 (uid2)")

===== G1 (uid) =====
Nodes           : 590,540
Directed edges  : 5,573,580
Average degree  : 9.44
Maximum degree  : 10
Isolated nodes  : 4,080

===== G2 (uid2) =====
Nodes           : 590,540
Directed edges  : 4,227,114
Average degree  : 7.16
Maximum degree  : 10
Isolated nodes  : 46,566



In [12]:
num_nodes = len(train)

indices = np.arange(num_nodes)

train_idx, valid_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=train["isFraud"]
)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
valid_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[train_idx] = True
valid_mask[valid_idx] = True

In [13]:
data_uid = Data(x=x, edge_index=edge_index_uid, y=y)

data_uid.train_mask = train_mask
data_uid.val_mask = valid_mask

In [14]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()

        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

        self.classifier = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        return self.classifier(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GraphSAGE(
    in_channels=data_uid.num_node_features,
    hidden_channels=64,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=5e-4,
)

criterion = torch.nn.CrossEntropyLoss()

train_loader = NeighborLoader(
    data_uid,
    input_nodes=data_uid.train_mask,
    num_neighbors=[10, 10],
    batch_size=2048,
    shuffle=True,
)

val_loader = NeighborLoader(
    data_uid,
    input_nodes=data_uid.val_mask,
    num_neighbors=[10, 10],
    batch_size=4096,
    shuffle=False,
)

In [15]:
for epoch in range(1, 21):
    # Train
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index)

        # Only the first batch.batch_size nodes are seed nodes
        loss = criterion(out[: batch.batch_size], batch.y[: batch.batch_size])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            probs = torch.softmax(out[: batch.batch_size], dim=1)[:, 1]
            all_probs.append(probs.cpu())
            all_labels.append(batch.y[: batch.batch_size].cpu())

    probs = torch.cat(all_probs).numpy()
    print(probs.min(), probs.max(), probs.mean())
    labels = torch.cat(all_labels).numpy()
    auc = roc_auc_score(labels, probs)

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {total_loss/len(train_loader):.4f} | "
        f"AUC {auc:.4f}"
    )

0.0 1.0 0.5308342
Epoch 01 | Loss 2542.0018 | AUC 0.5580
0.0 0.0 0.0
Epoch 02 | Loss 671.4328 | AUC 0.5000
0.0 1.0 0.1484249
Epoch 03 | Loss 392.1771 | AUC 0.5137
0.0 1.0 0.020044265
Epoch 04 | Loss 198.8714 | AUC 0.5370
0.0 1.0 0.01576946
Epoch 05 | Loss 75.5014 | AUC 0.5506
0.0 1.0 0.020911995
Epoch 06 | Loss 55.2877 | AUC 0.5585
0.0 1.0 0.041975774
Epoch 07 | Loss 48.5850 | AUC 0.5747
0.0 1.0 0.008418378
Epoch 08 | Loss 31.3448 | AUC 0.5773
0.0 1.0 0.00026164355
Epoch 09 | Loss 16.7005 | AUC 0.5014
0.0 1.0 0.025123825
Epoch 10 | Loss 16.4340 | AUC 0.6219
0.0 1.0 0.009041707
Epoch 11 | Loss 10.6733 | AUC 0.5187
0.0 1.0 0.026038373
Epoch 12 | Loss 10.0683 | AUC 0.5656
0.0 1.0 0.045985933
Epoch 13 | Loss 7.9492 | AUC 0.5934
0.0 1.0 0.030744141
Epoch 14 | Loss 4.8826 | AUC 0.6510
0.0 1.0 0.02334098
Epoch 15 | Loss 5.7068 | AUC 0.6744
0.0 1.0 0.013126106
Epoch 16 | Loss 3.4146 | AUC 0.6103
0.0 1.0 0.019821316
Epoch 17 | Loss 3.5749 | AUC 0.6014
0.0 1.0 0.0024031897
Epoch 18 | Loss 3.0879